In [1]:
# General
# OPTIONAL: if you want to have more information on what's happening, activate the logger as follows
import logging
#logging.basicConfig(level=logging.INFO)
import os
from pathlib import Path

# Visalization
import matplotlib.pyplot as plt

# Data
import numpy as np
import pandas as pd

# Embeddings and ML
import torch
from transformers import BertModel, BertTokenizer

/Users/martinjohannesnilsen/NTNU/Datateknologi/4. semester/code/.venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Create dataframes

In [2]:
from csv import QUOTE_NONE
import sys
import csv
csv.field_size_limit(sys.maxsize)

base_path = Path(os.path.abspath("")).parents[1] / "dataset_creation" / "data"
datasets = {
    "school_shooters": base_path / "school_shooters.csv",
    "manifestos": base_path / "manifestos.csv",
    "stair_twitter_archive": base_path / "stair_twitter_archive.csv",
    "twitter": base_path / "twitter.csv",
}

schoolshootersinfo_df = pd.read_csv(datasets["school_shooters"], encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)
manifesto_df = pd.read_csv(datasets["manifestos"], encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)
stair_twitter_archive_df = pd.read_csv(datasets["stair_twitter_archive"], encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)
twitter_df = pd.read_csv(datasets["twitter"], encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)
twitter_df

,date,text,name
0,NaN,Dorian Gray with Rainbow Scarf #LoveWins (from...,smile annotations
1,NaN,@SelectShowcase @Tate_StIves ... Replace with ...,smile annotations
2,NaN,@Sofabsports thank you for following me back. ...,smile annotations
3,NaN,@britishmuseum @TudorHistory What a beautiful ...,smile annotations
4,NaN,@NationalGallery @ThePoldarkian I have always ...,smile annotations
...,...,...,...
5052,2016-09-10,"""""""You bet Ben was belting louder than any gir...",umass
5053,2016-09-10,One of my hobby @ Ma Hood https://t.co/SHJDDWQ8QB,umass
5054,2016-09-10,Another Cardigan Records Hopscotch Day Party i...,umass
5055,2016-09-11,Bachelorette 💍💞 @ Laurita Winery https://t.co/...,umass


## Initial testing

One may argue that it could be beneficial to start with a overweigth of regular users as the school shooters posts will by its nature be a fraction. Therefore, I start with the school shooters info dataset (~1000 rows) against twitter (~5000 rows).

#### Embeddings intro

We need to represent text as matrices/vectors for machine learning pipelines to understand and find/learn patterns. This is where embeddings come into play as we are taking on the domain of text. For word embeddings, we want to encapsulate the similarity between words. As a rule of thumb, words that are similar should appear together in the grid/3d space. The same applies for sentence and document embeddings. Initially, we look further into word embeddings specifically.

The first approach is to use **word2vec**. This takes the cosine similarity but is not contextualized - this means that the word "bank" will have the same vectorial representation despite either meaning the place you havce your money or a long mound or slope, like a riverbank. **GloVe** is also context independent. The other alternatives, either **fasttext**, **elmo** or **BERT** is context dependent. 

Initially, we would like to use BERT embeddings as we want to utilize newer language models. Take a look at this [walkthrough](https://mccormickml.com/2019/05/14/BERT-word-embeddings-tutorial/).

#### Text preparation
We create a list of all texts

In [3]:
schoolshooters_texts = schoolshootersinfo_df["text"].to_list()

#### A note about token length

We would need to ensure that each run of BERT does not exceed 512 tokens. Chunking is the approach to take, and one could then choose one of multiple strategies. Chunking per post and taking the average seems like the way to go. This way, we keep each post to themselves, as we later on want to detect on a post basis, but could also chunk all texts of an author and then be able to run all texts of an author through BERT (then all text would be seen as a post, only a definition thing). 

In [4]:
# Load pre-trained model tokenizer (vocabulary)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Try on a concatenated string longer than 512 tokens
txt = " ".join(schoolshooters_texts[0:11])
tokens = tokenizer.encode_plus(
    txt, add_special_tokens=False
)
tokens

Token indices sequence length is longer than the specified maximum sequence length for this model (1054 > 512). Running this sequence through the model will result in indexing errors


{'input_ids': [2821, 1996, 8404, 1045, 2071, 2031, 2018, 11861, 2989, 2426, 2017, 2002, 5280, 5130, 1010, 2108, 8897, 2004, 2028, 1997, 2017, 1010, 2069, 2065, 2017, 2134, 1521, 1056, 6616, 1996, 2542, 4485, 2041, 1997, 2033, 1012, 2017, 2071, 2031, 2042, 2307, 1012, 1045, 2071, 2031, 2042, 2307, 1012, 3198, 4426, 2054, 2017, 2106, 2000, 2033, 2000, 2031, 2081, 2033, 4550, 1996, 12796, 1012, 2069, 2065, 2017, 2071, 2022, 1996, 6778, 1997, 2115, 16360, 2890, 10222, 19307, 1998, 10433, 6997, 1010, 2017, 3017, 13157, 1010, 2017, 2052, 2031, 26128, 1011, 19868, 2115, 4111, 23876, 2000, 6616, 2033, 1012, 2017, 2071, 2022, 2012, 2188, 2157, 2085, 5983, 2115, 8239, 6187, 9035, 2099, 1998, 2115, 8239, 2522, 16989, 2278, 1010, 2018, 2017, 2025, 10000, 13453, 15504, 2026, 3969, 1012, 2005, 2296, 2895, 1010, 2045, 2003, 2019, 5020, 1998, 4500, 4668, 1012, 2064, 2017, 2514, 1996, 3255, 2008, 2017, 21746, 2149, 1999, 1010, 2017, 8481, 1997, 16795, 1029, 2092, 1010, 2064, 2017, 2514, 2009, 1029, 203

#### Chunking
As we would like to utilize the embeddings with torch, we can return tensors instead of Python lists

In [5]:
tokens = tokenizer.encode_plus(
    txt, add_special_tokens=False, return_tensors="pt"
)
tokens

{'input_ids': tensor([[2821, 1996, 8404,  ..., 1055, 3268, 1012]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]])}

##### Explanation of approach in simplified example


In [6]:
# A tensor 10 numbers can be chunked
a = torch.arange(10)
print(a)
torch.split(a, 4)
print(a)

# And we can add tokens at front and back of each chunk
a = torch.cat([torch.Tensor([101]), a, torch.Tensor([102])])
print(a)

# Then we just need to pad the last chunk to be the right length
padding_len = 20 - a.shape[0]
if padding_len > 0:
    a = torch.cat(
        [a, torch.Tensor([0] * padding_len)]
    )
a

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
tensor([101.,   0.,   1.,   2.,   3.,   4.,   5.,   6.,   7.,   8.,   9., 102.])


tensor([101.,   0.,   1.,   2.,   3.,   4.,   5.,   6.,   7.,   8.,   9., 102.,
          0.,   0.,   0.,   0.,   0.,   0.,   0.,   0.])

#### Extracted method

In [9]:
from word_embeddings import get_word_embeddings
print(get_word_embeddings(txt))

Token indices sequence length is longer than the specified maximum sequence length for this model (1054 > 512). Running this sequence through the model will result in indexing errors


{'input_ids': tensor([[  101,  2821,  1996,  ...,  2000,  2017,   102],
        [  101, 26403,  2545,  ...,  2371,  1037,   102],
        [  101,  2309, 19471,  ...,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]], dtype=torch.int32)}
